# 03 — Prétraitement

Normalise les champs textuels (raison sociale, voies, communes) pour les 4 sources :
- EG FINESS  → `finess_eg_clean.parquet`   (suffixe `_eg`)
- EJ FINESS  → `finess_ej_clean.parquet`   (suffixe `_ej`)
- Etab SIRENE → `sirene_etab_clean.parquet` (suffixe `_etab`)
- UL SIRENE   → `sirene_ul_clean.parquet`   (suffixe `_ul`)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.pretraitement import pretraiter_eg, pretraiter_ej, pretraiter_etab, pretraiter_ul
from src.display       import afficher_tableau
from config.settings   import (
    FINESS_EG_RAW, FINESS_EJ_RAW, SIRENE_ETAB_RAW, SIRENE_UL_RAW,
    FINESS_EG_CLEAN, FINESS_EJ_CLEAN, SIRENE_ETAB_CLEAN, SIRENE_UL_CLEAN,
    PROCESSED_DIR,
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

import pyarrow.parquet as pq
import pyarrow as pa

## 1. EG FINESS

In [2]:
df_eg = pd.read_parquet(FINESS_EG_RAW)
df_eg_clean = pretraiter_eg(df_eg)
df_eg_clean.to_parquet(FINESS_EG_CLEAN, index=False)
print(f'EG : {len(df_eg_clean):,}  →  {FINESS_EG_CLEAN}')

afficher_tableau(
    df_eg_clean[['idstructure_stru', 'raisonsociale_norm_eg',
                 'libelle_voie_complet_eg', 'cdcommune_norm_eg',
                 'dept_eg', 'siren_eg']],
    'Aperçu EG prétraités',
)

EG : 104,805  →  /home/jovyan/work/projet_finess_sirene/data/processed/finess_eg_clean.parquet


idstructure_stru,raisonsociale_norm_eg,libelle_voie_complet_eg,cdcommune_norm_eg,dept_eg,siren_eg
2418252,INST SUP REEDUCATION PSYCHOMOTRICE,RUE FLEURY,03310,03,838296119
2418253,MAISON SANTE TROIS RIVIERES,PLACE PETITE VITESSE,07201,07,909193963
2418254,EML CENTRE IMAGERIE MEDICALE TOURNON,RUE ALPES,07324,07,440590198
2418255,MAISON SANTE HTES VALLEES ARDECHE,RUE DAME VENTADOUR,07156,07,879580538
2418256,MSP BUZANCY,RUE PETITE BAR,08089,08,848061230


## 2. EJ FINESS

In [3]:
df_ej = pd.read_parquet(FINESS_EJ_RAW)
df_ej_clean = pretraiter_ej(df_ej)
df_ej_clean.to_parquet(FINESS_EJ_CLEAN, index=False)
print(f'EJ : {len(df_ej_clean):,}  →  {FINESS_EJ_CLEAN}')

afficher_tableau(
    df_ej_clean[['idstructure_stru', 'nmsiren_stru', 'raisonsociale_norm_ej',
                 'libelle_voie_complet_ej', 'cdcommune_norm_ej', 'dept_ej']],
    'Aperçu EJ prétraités',
)

EJ : 54,185  →  /home/jovyan/work/projet_finess_sirene/data/processed/finess_ej_clean.parquet


idstructure_stru,nmsiren_stru,raisonsociale_norm_ej,libelle_voie_complet_ej,cdcommune_norm_ej,dept_ej
1831435,333483667,HABITAT PLURIEL,RUE ARMENY,13206,13
1831436,782974158,CANA,CHEMIN MADRAGUE VILLE,13215,13
1831438,334353471,REGIONALE INTEGRATION,RUE SAINT SEBASTIEN,13206,13
1831440,775559701,ENTRAIDE,RUE ROUX BRIGNOLES,13206,13
1831441,775558364,CAF 13,CHEMIN GIBBES,13214,13


## 3. Etab SIRENE

In [5]:
TAILLE_CHUNK = 500_000
writer = None
total = 0

pf = pq.ParquetFile(SIRENE_ETAB_RAW)
for i, batch in enumerate(pf.iter_batches(batch_size=TAILLE_CHUNK)):
    chunk_clean = pretraiter_etab(batch.to_pandas())
    table = pa.Table.from_pandas(chunk_clean, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(SIRENE_ETAB_CLEAN, table.schema)
    writer.write_table(table)
    total += len(chunk_clean)
    print(f"  chunk {i+1} — {total:,} lignes cumulées", end="\r")

if writer:
    writer.close()
print(f"\nEtab : {total:,}  →  {SIRENE_ETAB_CLEAN}")

afficher_tableau(
    pd.read_parquet(SIRENE_ETAB_CLEAN, columns=[
        'siret', 'denomination_norm_etab', 'enseigne1_norm_etab',
        'libelle_voie_complet_etab', 'code_commune_norm_etab', 'dept_etab'
    ]).head(),
    'Aperçu Etab prétraités',
)

  chunk 34 — 16,796,675 lignes cumulées
Etab : 16,796,675  →  /home/jovyan/work/projet_finess_sirene/data/processed/sirene_etab_clean.parquet


siret,denomination_norm_etab,enseigne1_norm_etab,libelle_voie_complet_etab,code_commune_norm_etab,dept_etab
00032517500065,,,RUE MARX DORMOY,13204,13
00542002100056,ETABLISSEMENTS LUCIEN BIQUEZ,,BOULEVARD PRES,80001,80
00542012000015,SOCIETE SUCRERIES MARQUENTERRE,,RUE FONTAINE,80688,80
00542012000023,SOCIETE SUCRERIES MARQUENTERRE,,ROUTE MONTREUIL,62688,62
00542012000056,SOCIETE SUCRERIES MARQUENTERRE,,CHEMIN GARENNES,80713,80


## 4. UL SIRENE

In [7]:
TAILLE_CHUNK = 500_000
writer = None
total = 0

pf = pq.ParquetFile(SIRENE_UL_RAW)
for i, batch in enumerate(pf.iter_batches(batch_size=TAILLE_CHUNK)):
    chunk_clean = pretraiter_ul(batch.to_pandas())
    table = pa.Table.from_pandas(chunk_clean, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(SIRENE_UL_CLEAN, table.schema)
    writer.write_table(table)
    total += len(chunk_clean)
    print(f"  chunk {i+1} — {total:,} lignes cumulées", end="\r")

if writer:
    writer.close()
print(f"\nUL : {total:,}  →  {SIRENE_UL_CLEAN}")

afficher_tableau(
    pd.read_parquet(SIRENE_UL_CLEAN, columns=[
        'siren', 'denomination_norm_ul', 'sigle_norm_ul',
        'libelle_voie_complet_ul', 'code_commune_norm_ul', 'dept_ul'
    ]).head(),
    'Aperçu UL prétraités',
)

  chunk 31 — 15,111,205 lignes cumulées
UL : 15,111,205  →  /home/jovyan/work/projet_finess_sirene/data/processed/sirene_ul_clean.parquet


siren,denomination_norm_ul,sigle_norm_ul,libelle_voie_complet_ul,code_commune_norm_ul,dept_ul
101753002,ND,ND,ND ND,05011,05
101753010,JSI,,RUE LIMOURS,78128,78
101753028,,,RUE JONQUILLES,34176,34
101753036,INDIVISION VERDAGUER RAMBEAU,,RUE FRERES COLIN,14118,14
101753044,ND,ND,ND ND,92035,92
